# Gold V1 — Google Colab Runner

Chạy **Gold Pipeline V1** trên Google Drive đã có sẵn snapshot **Silver V1**. Notebook **không** tải lại VTSNLP, không tạo lại Bronze/Silver, không sửa text và không huấn luyện mô hình.

**Thứ tự:** checkout nhánh Gold → cài package → mount đúng tài khoản Drive → verify Silver → kiểm tra đích xuất chưa tồn tại → chạy Gold CLI → kiểm tra checksum, số dòng, nhãn và giao nhau giữa các split.

**Lưu ý:** Gold V1 sử dụng tìm kiếm *candidate* near-duplicate theo heuristic, không phải kiểm tra mọi cặp văn bản. 50.000 văn bản có thể dùng nhiều CPU/RAM; khi bộ phát hiện candidate chạm giới hạn an toàn, pipeline sẽ dừng thay vì bỏ qua đối tượng nghi vấn. Chỉ chạy bước build **một lần** cho mỗi snapshot/cấu hình.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Ko4rut/vietnamese-multimodal-moderation.git"
BRANCH = __import__("os").environ.get("VTC_GIT_REF", "main")
PROJECT_DIR = Path("/content/vmm_gold_project")

# Repository công khai: không cần GitHub token/Colab Secret.
if (PROJECT_DIR / ".git").is_dir():
    dirty = subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "status", "--porcelain"],
        capture_output=True, text=True, check=True,
    )
    if dirty.stdout.strip():
        raise RuntimeError("Repo Colab có thay đổi chưa commit; hãy kiểm tra trước khi checkout")
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "checkout", "-B", BRANCH, "FETCH_HEAD"],
        check=True,
    )
else:
    if PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
        raise RuntimeError(f"{PROJECT_DIR} tồn tại và không phải Git repository")
    subprocess.run(
        ["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)],
        check=True,
    )

revision = subprocess.check_output(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
print(f"Repository: {BRANCH} @ {revision}")

In [ ]:
import sys
import subprocess

# Gold chỉ sử dụng dependencies cơ bản trong pyproject.toml (không cần [data]/[eda]).
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "."],
    cwd=str(PROJECT_DIR), check=True,
)

# Đảm bảo kernel hiện tại tìm thấy package src-layout ngay sau khi pip install.
SRC_DIR = (PROJECT_DIR / "src").resolve()
if not (SRC_DIR / "vietnamese_topic_classification" / "__init__.py").is_file():
    raise FileNotFoundError(f"Không tìm thấy Python package: {SRC_DIR}")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from vietnamese_topic_classification.config import get_domain_labels
from vietnamese_topic_classification.data.gold import GoldConfig, GoldPipeline, GoldReader
from vietnamese_topic_classification.data.silver import SilverReader

print("Python:", sys.version.split()[0])
print("Number of topic labels:", len(get_domain_labels()))
print("Gold package ready.")
from vietnamese_topic_classification.data.colab_lake import resolve_mounted_lake


In [ ]:
from google.colab import drive

MOUNT = "/content/vtsnlp_drive"
drive.mount(MOUNT)

# Hãy chọn đúng Google Drive chứa Data Lake hiện tại.
DATA_LAKE_ROOT = resolve_mounted_lake(MOUNT)
SILVER_REPORT_NAME = "silver_quality_v1_b81fcce589_seed42_buf10000_n50000.json"

if not DATA_LAKE_ROOT.is_dir():
    raise FileNotFoundError(
        f"Không tìm thấy Data Lake: {DATA_LAKE_ROOT}. "
        "Hãy kiểm tra tài khoản Drive hoặc thay DATA_LAKE_ROOT."
    )

report_path = DATA_LAKE_ROOT / "05_reports" / "silver" / "v1" / SILVER_REPORT_NAME
if not report_path.is_file():
    examples = sorted(p.name for p in (DATA_LAKE_ROOT / "05_reports" / "silver" / "v1").glob("silver_quality_*.json"))
    raise FileNotFoundError(
        f"Không tìm thấy report {report_path}. Silver reports hiện có: {examples[:20]}"
    )

silver = SilverReader(data_lake_root=DATA_LAKE_ROOT, metadata_name=SILVER_REPORT_NAME)
silver_verified = silver.verify()  # Xác minh SHA256 và row count; không thay đổi Silver.
print("Verified Silver:", silver_verified)
print("Silver JSONL:", silver.data_path)
print("Silver report:", silver.report_path)

In [ ]:
# Cấu hình Gold cố định: thay đổi tại đây trước khi build, nếu thật sự cần.
SEED = 42
TRAIN_RATIO = 0.80
VALIDATION_RATIO = 0.10
TEST_RATIO = 0.10
SIMILARITY_THRESHOLD = 0.90
MAX_CANDIDATES = 2000

gold_config = GoldConfig(
    seed=SEED,
    train_ratio=TRAIN_RATIO,
    validation_ratio=VALIDATION_RATIO,
    test_ratio=TEST_RATIO,
    similarity_threshold=SIMILARITY_THRESHOLD,
    max_candidates_per_record=MAX_CANDIDATES,
)
gold = GoldPipeline(silver, config=gold_config)
OUTPUT_FILES = gold.planned_paths(silver_verified["sha256"])

# Không cho phép ghi nhầm 03_gold hoặc bất kỳ thư mục Gold ngoài chuẩn.
gold_root = DATA_LAKE_ROOT / "04_gold"
if (
    any(OUTPUT_FILES[split].parent != gold_root for split in ("train", "validation", "test"))
    or OUTPUT_FILES["manifest"].parent != gold_root / "manifests"
    or OUTPUT_FILES["report"].parent != DATA_LAKE_ROOT / "05_reports"
):
    raise RuntimeError("Gold output paths không khớp kiến trúc 04_gold/05_reports")

# Nếu snapshot đã được chuyển đúng chỗ thì chỉ xác minh và bỏ qua build.
stale_uploads = [
    path.with_name(path.name + ".uploading") for path in OUTPUT_FILES.values()
    if path.with_name(path.name + ".uploading").exists()
]
if stale_uploads:
    raise FileExistsError("Gold có file .uploading chưa hoàn tất: " + ", ".join(map(str, stale_uploads)))

existing = [path for path in OUTPUT_FILES.values() if path.exists()]
GOLD_ALREADY_EXISTS = False
if existing:
    if len(existing) != len(OUTPUT_FILES) or not all(path.is_file() for path in existing):
        raise FileExistsError(
            "Gold chỉ tồn tại một phần; cần kiểm tra thủ công, không tự rebuild: "
            + ", ".join(map(str, existing))
        )
    existing_reader = GoldReader(DATA_LAKE_ROOT, OUTPUT_FILES["manifest"].name)
    existing_verified = existing_reader.verify()
    if existing_reader.manifest["source"]["silver_sha256"] != silver_verified["sha256"]:
        raise AssertionError("Gold hiện có không khớp Silver nguồn")
    GOLD_ALREADY_EXISTS = True
    print("EXISTING GOLD VERIFIED — sẽ bỏ qua bước build:", existing_verified)
else:
    print("PREFLIGHT PASSED — chưa có Gold output trùng snapshot/cấu hình.")
    
for name, path in OUTPUT_FILES.items():
    print(f"{name:10s}", path)

In [ ]:
# Chạy CLI gốc thay vì viết lại thuật toán split trong notebook.
import os

command = [
    sys.executable, str(PROJECT_DIR / "scripts/05_build_gold.py"),
    "--data-lake-root", str(DATA_LAKE_ROOT),
    "--report-name", SILVER_REPORT_NAME,
    "--seed", str(SEED),
    "--train-ratio", str(TRAIN_RATIO),
    "--validation-ratio", str(VALIDATION_RATIO),
    "--test-ratio", str(TEST_RATIO),
    "--similarity-threshold", str(SIMILARITY_THRESHOLD),
    "--max-candidates", str(MAX_CANDIDATES),
]
env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join(
    path for path in [str(SRC_DIR), env.get("PYTHONPATH", "")] if path
)
if GOLD_ALREADY_EXISTS:
    print("GOLD V1 ALREADY EXISTS — SKIPPING BUILD.")
else:
    subprocess.run(command, cwd=str(PROJECT_DIR), env=env, check=True)
    print("GOLD V1 BUILD COMPLETED.")

In [ ]:
# Kiểm tra toàn bộ Gold snapshot đã xuất ra Drive, không rebuild.
import json
from collections import Counter

for name, path in OUTPUT_FILES.items():
    if not path.is_file():
        raise FileNotFoundError(f"Thiếu Gold artifact: {name}: {path}")

reader = GoldReader(DATA_LAKE_ROOT, OUTPUT_FILES["manifest"].name)
verified = reader.verify()  # Tự kiểm tra SHA256 và số dòng của 3 JSONL.
manifest = reader.manifest
report = json.loads(OUTPUT_FILES["report"].read_text(encoding="utf-8"))

if manifest["source"]["silver_sha256"] != silver_verified["sha256"]:
    raise AssertionError("Gold manifest không khớp source Silver SHA256")
if report["input_rows"] != silver_verified["records"]:
    raise AssertionError("Gold quality report không khớp số dòng Silver")
if verified["records"] != silver_verified["records"]:
    raise AssertionError("Tổng rows Gold không bằng số rows Silver")

all_ids = set()
per_split_counts = {}
label_to_id = manifest["label_to_id"]
if len(label_to_id) != 25:
    raise AssertionError("Gold phải có đúng 25 label IDs")

for split in ("train", "validation", "test"):
    domain_counts = Counter()
    examples = []
    for row in reader.iter_split(split):
        row_id = row["id"].strip()
        if row_id in all_ids:
            raise AssertionError(f"Duplicate ID giữa các split: {row_id}")
        all_ids.add(row_id)
        if row["label_id"] != label_to_id[row["domain"]]:
            raise AssertionError(f"Label mapping sai ở {split}: {row_id}")
        if row["text_length"] != len(row["text"]):
            raise AssertionError(f"text_length sai ở {split}: {row_id}")
        domain_counts[row["domain"]] += 1
        if len(examples) < 1:
            examples.append({"id": row["id"], "domain": row["domain"],
                             "label_id": row["label_id"], "preview": row["text"][:140]})
    if sum(domain_counts.values()) != verified["split_rows"][split]:
        raise AssertionError(f"Split count mismatch: {split}")
    per_split_counts[split] = domain_counts
    print(f"{split.upper()}: {verified['split_rows'][split]:,} records, "
          f"{len(domain_counts)} topics. Sample:", examples)

if len(all_ids) != verified["records"]:
    raise AssertionError("Gold IDs không khớp tổng record count")

print("\nGOLD V1 VERIFICATION PASSED")
print("Train/Validation/Test:", verified["split_rows"])
print("Near-duplicate audit:", report["near_duplicate_audit"])
print("Manifest:", OUTPUT_FILES["manifest"])
print("Quality report:", OUTPUT_FILES["report"])
print("Lưu ý: near-duplicate search là heuristic (không exhaustive).")

## Kết quả và bước tiếp theo

Khi cell cuối hiện **GOLD V1 VERIFICATION PASSED**, bạn có thể dùng \`GoldReader.iter_split("train")\`, \`iter_split("validation")\` và \`iter_split("test")\` để bắt đầu Modeling. TF-IDF/Tokenizer chỉ được fit hoặc áp dụng phù hợp trong Modeling.

Dữ liệu được publish dưới \`04_gold/\` và \`05_reports/\`. Nếu Colab ngắt giữa chừng, **không xóa file mù quáng**: kiểm tra \`.uploading\`, manifest và ba snapshot trước khi quyết định chạy lại. Notebook không hỗ trợ ghi đè Gold.